## Text Summarization using BART on Samsum dataset.

### Import Dependencies

In [4]:
from datasets import load_dataset
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Trainer , TrainingArguments
import evaluate

### Load the dataset

In [26]:
dataset = load_dataset("knkarthick/samsum")

In [27]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

### Load the model and tokenizer

In [7]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [8]:
model_checkpoint = 'facebook/bart-large-cnn'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

### Try the model without fine tuning

In [9]:
pipe = pipeline(
    "summarization" , model = model_checkpoint, device = device
)

Device set to use cuda


In [10]:
dialouge_1 = dataset['test'][0]['dialogue']
print(dialouge_1)

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye


In [11]:
outputs = pipe(dialouge_1 , max_length = 20, min_length = 10, do_sample = False)
outputs

[{'summary_text': "Hannah asks Amanda for Betty's phone number. Amanda can't find it. Hannah"}]

In [12]:
print(dataset['test'][0]['summary'])

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


## Start Fine Tuning

### Define the tokenization function

In [29]:
def tokenization(batch):
    # get the dialouge and summary
    summaries = batch['summary']
    dialouges = batch['dialogue']
    
    # do tokenization on dialouge
    input_encoded = tokenizer(
        dialouges,
        padding = False,
        max_length = 256,
        truncation = True
    )
    # do tokenization on summary
    labels = tokenizer(
        summaries,
        padding = False,
        max_length = 128,
        truncation = True
    )
    # replace pad token id with -100 for loss masking
    labels['input_ids'] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]
    
    return {
        'input_ids': input_encoded['input_ids'],
        'attention_mask': input_encoded['attention_mask'],
        'labels': labels['input_ids']
    }

In [22]:
tokenizer.pad_token_id

1

In [30]:
tokenized_dataset = dataset.map(
    tokenization,
    batched = True,
    remove_columns = dataset['train'].column_names
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [32]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

### Define training arguments

In [33]:
training_args = TrainingArguments(
    output_dir = "./BART-Summarization",
    overwrite_output_dir = True,
    eval_strategy = 'epoch',
    per_device_eval_batch_size = 4,
    per_device_train_batch_size = 4,
    num_train_epochs = 2,
    remove_unused_columns = True,
    report_to = None, 
    learning_rate = 2e-5,
    warmup_steps = 500,
    weight_decay = 0.01,
)

### Define the Trainer and start training

In [34]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model.to(device)
)

In [35]:
trainer = Trainer(
    model = model.to(device),
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
)

In [36]:
trainer.train()

Step,Training Loss
500,1.648700
1000,1.567400
1500,1.499400
2000,1.347600
2500,1.056800
3000,1.047100
3500,1.047100
4000,0.823600
4500,0.690800
5000,0.706800


c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=5526, training_loss=1.0982064689970343, metrics={'train_runtime': 1729.1933, 'train_samples_per_second': 25.557, 'train_steps_per_second': 3.196, 'total_flos': 2.278671548959949e+16, 'train_loss': 1.0982064689970343, 'epoch': 3.0})

### Test the model

In [37]:
eval_results = trainer.evaluate(
    eval_dataset = tokenized_dataset['test']
)
eval_results

{'eval_loss': 1.7061970233917236,
 'eval_runtime': 9.7637,
 'eval_samples_per_second': 83.882,
 'eval_steps_per_second': 10.549,
 'epoch': 3.0}

### save the model and tokenizer

In [38]:
%pwd

'd:\\Tipto\\Natural-Language-Processing\\Text Summarization'

In [ ]:
model.save_pretrained("/BART-Model")
tokenizer.save_pretrained("/BART-Tokenizer")

### Load the model and use for inference

In [48]:
device

device(type='cuda')

In [64]:
tokenizer = AutoTokenizer.from_pretrained("/BART-Tokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained("/BART-Model").to(device)

c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\transformers\models\bart\configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


In [65]:
from transformers import GenerationConfig
generation_config = GenerationConfig(
    max_length=150,
    min_length=40,
    num_beams=4,
    num_return_sequences=1,
    length_penalty=2.0,
    early_stopping=True,
    forced_bos_token_id=0,  
    decoder_start_token_id=tokenizer.bos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [66]:
model.generation_config = generation_config

In [67]:
def summarize(text):
    inputs = tokenizer(
        text, 
        max_length=1024, 
        truncation=True, 
        return_tensors='pt'
    ).to(device)
    
    # generate the summary - will use model.generation_config
    summary_ids = model.generate(**inputs)
    
    # decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [68]:
text = """As Yogi Berra famously said, its tough to make predictions, especially about the future. But had the baseball legend spent any time observing the UN climate negotiations, he could have safely predicted that climate finance will prove to be a key sticking point at COP29 in Baku at the end of this year.  

Who will pay and how much? are perennial questions at the climate talks, but this year, the discussions about climate finance will be especially prominent. At COP29, Parties to the Paris Agreement must negotiate a new climate finance goal, to replace the existing commitment from 2009 for developed countries to provide US$100 billion climate finance annually from 2020 to 2025 - a commitment that only in 2022 was starting to be fulfilled, according to a recent OECD report. 

It is vital that the forthcoming Bonn Climate Change Conference sends the right political signals, and lays the procedural and technical groundwork for an ambitious climate finance deal in Baku.

A pressing need

With global warming already destabilising the climate and devastating peoples lives and livelihoods, the need for finance to reduce greenhouse gas emissions and to adapt to a warming world has never been more pressing.

The sums involved are large. The Paris Agreements Global Stocktake process estimates that US$5.8-5.9 trillion is required to implement Nationally Determined Contributions (NDCs) in developing countries up to 2030. They will require US$215-387 billion annually over this period for adaptation. Investments of US$1.5 trillion in renewable energy are required worldwide every year up until 2030, according to IRENA.

But these sums are also affordable and beneficial for developed countries. They should be seen in the context of ongoing investments in energy and other infrastructure: around US$2.3 trillion was invested in energy infrastructure in 2023, of which US$1.74 trillion was in clean energy. These investments will generate strong returns for their investors and reduce the costs for energy consumers.

And, crucially, they should also be seen in the context of the alternative. The latest research estimates that the world economy is already set to face a 19% income reduction within the next 26 years based on the levels of warming we have already locked in. The more we delay and the more the planet heats, the greater the economic costs will be.

Laying the foundations for a new finance goal

While financial resources are beginning to flow, they are not flowing fast enough, and certainly not flowing to those developing countries where need is greatest and access to finance is most challenging.

The UN climate framework provides mechanisms that can enable those flows of climate finance. Back in 2015, parties at the climate talks agreed to establish a “new collective quantified goal” (NCQG) for climate finance. They agreed that the NCQG would be set prior to 2025.

The  ultimate size of the NCQG will be a product of the negotiations, but Parties have agreed it must be a significant increase from the floor of US$100 billion annually. For WWF, it must be needs-based and sufficiently ambitious to meet the scale of the challenge we face, and immediately accessible to help countries that are already facing the chaos of a destabilised climate system.

While developed countries are expected to provide financial and technical support, developing countries also have a role to play. Parties are due to submit revised NDCs in 2025, presenting how they plan to reduce emissions and adapt to climate change. Developing countries have the opportunity to use their NDCs to set out how international climate finance can support them and increase their ambition. To do this, they need to know the finance will be forthcoming."""

In [ ]:
summary = summarize(text=text)
print(f"\nFinal Summary: {summary}")